# Diffusion models

_Modern Deep Learning & AI_

**Destroy a picture with noise step by step, then teach a network to undo it. Run the undo from pure noise to create.**

A diffusion model learns to generate by first learning to destroy, then reversing it.

     The forward process takes a clean image and adds a little Gaussian noise, again and again, until it is pure static. This part is fixed, no learning needed.

---

This notebook is a practice scaffold for the **Diffusion models** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

T = 200
betas = torch.linspace(1e-4, 0.02, T)          # noise schedule
alpha_bar = torch.cumprod(1.0 - betas, dim=0)  # product of (1 - beta) up to t

# tiny noise-predictor: takes x_t and a (scaled) timestep, predicts the noise
net = nn.Sequential(nn.Linear(5 + 1, 32), nn.ReLU(), nn.Linear(32, 5))

torch.manual_seed(0)
x0 = torch.rand(8, 5)                           # clean synthetic data
t = torch.randint(0, T, (8,))                   # a random step per sample
ab = alpha_bar[t].unsqueeze(1)                  # (8, 1)
eps = torch.randn_like(x0)                      # the noise we add
x_t = ab.sqrt() * x0 + (1 - ab).sqrt() * eps    # closed-form forward jump

t_feat = (t.float() / T).unsqueeze(1)           # timestep as a feature
eps_pred = net(torch.cat([x_t, t_feat], dim=1)) # predict the added noise
loss = F.mse_loss(eps_pred, eps)                # train to name the noise
loss.backward()
print("x_t shape:", x_t.shape, "loss:", round(loss.item(), 4))

## Visualize the data & results

_Apply the real DDPM noise schedule to one real digit-3 image: how fast does signal turn into noise?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# forward diffusion of a REAL digit image with the standard DDPM schedule
digits = load_digits()
img = (digits.data / 16.0)[np.where(digits.target == 3)[0][0]]  # one real "3"
T = 1000
betas = np.linspace(1e-4, 0.02, T)              # linear noise schedule
alpha_bar = np.cumprod(1.0 - betas)
ts = np.arange(0, T, 100)
signal = np.sqrt(alpha_bar[ts])
noise = np.sqrt(1.0 - alpha_bar[ts])

rng = np.random.default_rng(3)
eps = rng.standard_normal(img.shape)
meas = []
for t in ts:
    xt = np.sqrt(alpha_bar[t]) * img + np.sqrt(1 - alpha_bar[t]) * eps
    meas.append(np.var(np.sqrt(1 - alpha_bar[t]) * eps) / (np.var(xt) + 1e-9))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ts, signal, color="#4ea1ff", label="signal sqrt(alpha_bar)")
ax.plot(ts, noise, color="#ff7b72", label="noise sqrt(1-alpha_bar)")
ax.plot(ts, np.clip(meas, 0, 1), color="#ffb454", label="measured noise fraction")
ax.set_xlabel("timestep t"); ax.set_ylabel("fraction")
ax.set_title("real digit image dissolving into noise"); ax.legend()
plt.show()